# Fase 4 – Natural Language Processing: chatbot académico RAG

**Objetivo:** documentar y ejecutar el backend del orientador académico, validar recuperación documental y generación local con Qwen, y registrar una evaluación de diez consultas.

> Ejecuta las celdas en orden. Las respuestas y métricas deben conservarse tal como se produzcan en tu equipo; este notebook no incluye resultados simulados.


## 1. Instalación y configuración

Instala las dependencias desde `requirements.txt` en el entorno del proyecto. La descarga inicial de modelos requiere Internet y puede tardar.


In [ ]:
# Si el entorno aún no tiene las dependencias, descomenta y ejecuta:
# %pip install -r ../requirements.txt

from pathlib import Path
import os, sys, time, csv
import pandas as pd

PROJECT = Path.cwd()
if not (PROJECT / "rag_engine.py").exists():
    PROJECT = PROJECT.parent
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
print("Directorio del proyecto:", PROJECT)
print("Documentos disponibles:", list((PROJECT / "knowledge").glob("*.txt")))


## 2. Carga del modelo de embeddings e indexación persistente

Sentence-Transformers transforma cada fragmento en un vector multilingüe normalizado. Los vectores se guardan como BLOB en SQLite para que el índice permanezca disponible entre ejecuciones.


In [ ]:
from rag_engine import (get_embedder, index_documents, retrieve, list_documents,
                        get_session_summary, save_session_summary, LLM_MODEL)

start = time.time()
embedder = get_embedder()
index_stats = index_documents(embedder)
print("Resumen de indexación:", index_stats)
print("Tiempo de carga e indexación (segundos):", round(time.time() - start, 2))


## 3. Verificación de recuperación (RAG)

Se consulta la base documental y se inspeccionan los fragmentos recuperados y su puntuación de similitud.


In [ ]:
pregunta_demo = "¿Cómo organizo mi estudio autónomo?"
fragmentos = retrieve(pregunta_demo, embedder=embedder)
for item in fragmentos:
    print(f"Fuente: {item['source']} | Fragmento: {item['chunk'] + 1} | Score: {item['score']:.4f}")
    print(item['text'], "\n")


## 4. Carga del LLM abierto Qwen y generación

El tokenizer aplica la plantilla conversacional del modelo. La respuesta se genera en CPU para favorecer la compatibilidad. El primer inicio descarga el modelo y puede ser lento.


In [ ]:
from transformers import AutoTokenizer, pipeline

print("Modelo de lenguaje:", LLM_MODEL)
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
generator = pipeline("text-generation", model=LLM_MODEL, tokenizer=tokenizer, device=-1)
print("Tiempo de carga del LLM (segundos):", round(time.time() - t0, 2))


In [ ]:
from rag_engine import generate_answer

respuesta_demo = generate_answer(
    pregunta_demo, fragmentos, history=[], summary="",
    generator=generator, tokenizer=tokenizer
)
print(respuesta_demo)


## 5. Memoria conversacional con SQLite

Se prueba el almacenamiento de un resumen de sesión. No se deben introducir datos personales o sensibles.


In [ ]:
session_test = "notebook-fase4-demo"
save_session_summary(session_test, "El usuario consulta recomendaciones para organizar el estudio autónomo.")
print("Resumen recuperado:", get_session_summary(session_test))


## 6. Evaluación de diez consultas

Ejecuta las consultas, revisa cada respuesta y completa las valoraciones manuales en el CSV. La respuesta generada se guarda como evidencia; los puntajes deben asignarse después de leerla y contrastarla con la fuente.


In [ ]:
test_queries = [
    ("T01", "¿Cómo organizo mi estudio autónomo?"),
    ("T02", "¿Qué reviso antes de entregar un trabajo?"),
    ("T03", "¿Cómo participo en un foro académico?"),
    ("T04", "¿Cómo puedo evitar el plagio?"),
    ("T05", "¿Qué hago si no entiendo una consigna?"),
    ("T06", "¿Cuál es el nombre de mi docente?"),
    ("T07", "¿Cuál es la fecha exacta de entrega?"),
    ("T08", "¿Qué universidad ofrece esta guía?"),
    ("T09", "Dame un enlace oficial para solicitar una prórroga."),
    ("T10", "¿Qué norma APA debo usar para las referencias?"),
]

rows = []
for test_id, question in test_queries:
    t0 = time.time()
    hits = retrieve(question, embedder=embedder)
    if hits:
        response = generate_answer(question, hits, history=[], summary="", generator=generator, tokenizer=tokenizer)
    else:
        response = "No se recuperó evidencia suficiente en la base documental."
    elapsed = round(time.time() - t0, 2)
    sources = "; ".join(sorted({h["source"] for h in hits}))
    rows.append({"ID": test_id, "Pregunta": question, "Respuesta_observada": response,
                 "Fuente_recuperada": sources, "Tiempo_segundos": elapsed})
    print("\n", test_id, question, "\nRespuesta:", response, "\nFuentes:", sources, "\nTiempo:", elapsed)

results = pd.DataFrame(rows)
results.to_csv(PROJECT / "outputs" / "respuestas_observadas.csv", index=False, encoding="utf-8-sig")
results


## 7. Registro de evaluación cualitativa

Abre `evaluacion_10_consultas.csv` y registra para cada pregunta claridad, relevancia, utilidad, satisfacción y observaciones. No asignes puntuaciones automáticas sin revisión humana.


In [ ]:
print("Plantilla de evaluación:", PROJECT / "evaluacion_10_consultas.csv")
print("Respuestas observadas:", PROJECT / "outputs" / "respuestas_observadas.csv")


## 8. Conclusiones y análisis de errores

Después de ejecutar las pruebas, redacta aquí tres fallos observados, su posible causa y una mejora propuesta. Considera, por ejemplo: recuperación de fragmentos irrelevantes, respuestas que agregan información no respaldada y lentitud de inferencia. Solo registra como hallazgos los problemas que efectivamente observes.


In [ ]:
# Completar después de ejecutar y revisar las diez pruebas.
error_analysis = [
    {"problema_observado": "", "evidencia_ID": "", "causa_probable": "", "mejora_propuesta": ""},
    {"problema_observado": "", "evidencia_ID": "", "causa_probable": "", "mejora_propuesta": ""},
    {"problema_observado": "", "evidencia_ID": "", "causa_probable": "", "mejora_propuesta": ""},
]
error_analysis
